# 12. クロスインパクト分析（Cross-Impact Analysis） — 練習問題

**対象技術**: 量子コンピューティング

クロスインパクト分析は、複数の将来事象が互いの発生確率に与える影響をクロスインパクト行列で表現し、モンテカルロ法で「事象を逐次抽選し、確定するたびに未確定事象の確率を更新する」ことで、相互作用を織り込んだ整合的な同時確率分布を得る手法である。このノートブックでは量子関連の5将来事象を題材に、修正後の周辺確率と特定シナリオの発生確率を算出する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 将来事象とクロスインパクト行列の定義

量子関連の5将来事象とその初期（周辺）確率、およびオッズ比方式のクロスインパクト行列を定義する。行列の要素 `M[i][j]` は「事象 i が発生したとき事象 j の発生オッズに掛ける倍率」を表す。

In [ ]:
EVENTS = [
    "E1: CRQCが2035年までに登場",
    "E2: PQC標準の調達義務化が成立",
    "E3: 量子クラウド市場で寡占が成立",
    "E4: 量子人材の深刻な枯渇",
    "E5: 大規模な暗号侵害インシデント発生",
]

# 各事象の初期(周辺)確率 = 他事象の影響を考えない単独の発生確率。
PRIOR = np.array([0.40, 0.55, 0.45, 0.50, 0.30])

# クロスインパクト行列(オッズ比方式)。
# M[i][j] = 事象iが発生したとき事象jの発生オッズに掛ける倍率。
M = np.array([
    # ->E1   ->E2   ->E3   ->E4   ->E5
    [1.00,  3.00,  1.20,  1.00,  2.50],   # E1(CRQC登場)が発生したとき
    [1.00,  1.00,  1.10,  1.00,  0.70],   # E2(PQC義務化)が発生したとき
    [1.00,  1.00,  1.00,  0.70,  1.00],   # E3(寡占)が発生したとき
    [0.50,  1.00,  1.30,  1.00,  1.00],   # E4(人材枯渇)が発生したとき
    [1.00,  2.00,  1.00,  1.00,  1.00],   # E5(暗号侵害)が発生したとき
])

print("[将来事象と初期(周辺)確率]")
print("-" * 70)
for name, p in zip(EVENTS, PRIOR):
    print(f"  {name:<32} 初期確率={p:.2f}")

## 確率とオッズの相互変換

クロスインパクトはオッズ比で表現されるため、確率↔オッズの変換関数を用意する。オッズ比方式は更新後の値が必ず 0〜1 に収まる。

In [ ]:
def prob_to_odds(p):
    """確率をオッズ(p / (1-p))に変換する。"""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return p / (1.0 - p)


def odds_to_prob(o):
    """オッズを確率に戻す。値は必ず 0〜1 に収まる。"""
    return o / (1.0 + o)


# 動作確認
for p in [0.1, 0.5, 0.9]:
    o = prob_to_odds(p)
    print(f"  p={p:.1f} -> odds={o:.3f} -> p={odds_to_prob(o):.3f}")

## モンテカルロ・クロスインパクト・アルゴリズム

1試行 = 全事象が確定するまで、未確定事象から1つランダムに選び、現在の確率で発生/不発生を抽選し、発生したらその事象から出るクロスインパクトを未確定事象すべてに適用する、という手続きを繰り返す。

In [ ]:
def run_cross_impact(prior, matrix, n_trials, rng):
    """モンテカルロ・クロスインパクト・アルゴリズム。
    戻り値: 各試行の発生パターン(0/1)を並べた配列 (n_trials, n_events)。
    """
    n = len(prior)
    outcomes = np.zeros((n_trials, n), dtype=int)
    for t in range(n_trials):
        current = prior.copy()          # この試行での現在確率
        undecided = list(range(n))      # 未確定事象のインデックス
        rng.shuffle(undecided)          # 処理順序もランダム化
        result = np.zeros(n, dtype=int)
        while undecided:
            i = undecided.pop()
            occurred = rng.random() < current[i]
            result[i] = 1 if occurred else 0
            if occurred:
                # 事象iが発生 -> 未確定事象jのオッズに M[i][j] を乗じる
                for j in undecided:
                    o = prob_to_odds(current[j]) * matrix[i, j]
                    current[j] = odds_to_prob(o)
        outcomes[t] = result
    return outcomes


rng = np.random.default_rng(42)
n_trials = 20000
outcomes = run_cross_impact(PRIOR, M, n_trials, rng)
revised = outcomes.mean(axis=0)  # 修正後の周辺確率
print(f"モンテカルロ {n_trials} 試行を完了しました。")

## 修正後の周辺確率

相互作用を織り込んだ結果、各事象の発生確率が初期確率からどれだけ動いたかを見る。

In [ ]:
print(f"[修正後の周辺確率] (モンテカルロ {n_trials} 試行)")
print("  相互作用を織り込んだ結果、初期確率からどれだけ動いたか")
print("-" * 70)
for i, name in enumerate(EVENTS):
    diff = revised[i] - PRIOR[i]
    arrow = "上昇↑" if diff > 0.01 else ("低下↓" if diff < -0.01 else "ほぼ不変")
    print(f"  {name:<32} {PRIOR[i]:.2f} -> {revised[i]:.2f} "
          f"(差 {diff:+.3f} {arrow})")

## シナリオ発生確率の算出

特定の事象の組合せ＝シナリオの同時発生確率を、整合的同時分布から読み取る。独立を仮定した素朴な積と比較する。

In [ ]:
def scenario_probability(outcomes, conditions):
    """特定シナリオの同時発生確率を同時分布から読み取る。
    conditions: {事象index: 0または1} の辞書。
    """
    mask = np.ones(len(outcomes), dtype=bool)
    for idx, val in conditions.items():
        mask &= (outcomes[:, idx] == val)
    return mask.mean()


# 最悪シナリオ: E1(CRQC登場) かつ E5(暗号侵害) が両方発生
joint = scenario_probability(outcomes, {0: 1, 4: 1})
naive = PRIOR[0] * PRIOR[4]  # 独立を仮定した素朴な積

print("[シナリオ発生確率] 最悪シナリオ「CRQC登場 かつ 大規模暗号侵害」")
print("-" * 70)
print(f"  独立仮定の素朴な積       : {naive:.4f}  (P(E1)×P(E5))")
print(f"  クロスインパクト同時確率 : {joint:.4f}  (整合的同時分布から)")
print(f"  差                       : {joint - naive:+.4f}")
print("  -> E1がE5を引き上げる正の連関のぶん、最悪シナリオは素朴な積より")
print("     起こりやすい。相互作用を無視した評価はこのリスクを過小評価する。")

good = scenario_probability(outcomes, {1: 1, 4: 0})
print(f"\n[参考] 「PQC義務化が成立 かつ 暗号侵害が起きない」確率 = {good:.4f}")

## 可視化1: 修正前後の周辺確率の比較

各事象の初期（修正前）確率とモンテカルロ後の修正後確率を、並べた棒グラフで比較する。相互作用がどの事象の確率を押し上げ/押し下げたかが一目で分かる。

In [ ]:
labels = [f"E{i+1}" for i in range(len(EVENTS))]
x = np.arange(len(EVENTS))
w = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w/2, PRIOR, w, label="Prior (initial)", color="#7fb8d6")
ax.bar(x + w/2, revised, w, label="Revised (post-MC)", color="#1f3f5c")
for i in range(len(EVENTS)):
    ax.text(x[i] - w/2, PRIOR[i] + 0.01, f"{PRIOR[i]:.2f}",
            ha="center", fontsize=8)
    ax.text(x[i] + w/2, revised[i] + 0.01, f"{revised[i]:.2f}",
            ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Marginal probability")
ax.set_ylim(0, 0.8)
ax.set_title("Cross-impact: prior vs revised marginal probabilities")
ax.legend()
fig.tight_layout()
plt.show()

## 可視化2: クロスインパクト行列のヒートマップ

クロスインパクト行列を `imshow` で表示する。1.0 より大きいセル（正の連関）と小さいセル（負の連関）を発散カラーマップで描く。

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M, cmap="RdBu_r", vmin=0, vmax=3)
ticks = [f"E{i+1}" for i in range(len(EVENTS))]
ax.set_xticks(range(len(EVENTS)))
ax.set_xticklabels(ticks)
ax.set_yticks(range(len(EVENTS)))
ax.set_yticklabels(ticks)
ax.set_xlabel("affected event j")
ax.set_ylabel("occurring event i")
for i in range(len(EVENTS)):
    for j in range(len(EVENTS)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color="#222222")
ax.set_title("Cross-impact matrix M[i][j] (odds multiplier)")
fig.colorbar(im, ax=ax, label="odds multiplier (1.0 = no effect)")
fig.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文においてクロスインパクト分析は、複数の将来事象が互いに独立だという素朴な前提を解除する装置として用いられる。論文はまず各事象の初期確率を見積もり、事象間の促進・抑制関係をクロスインパクト行列として与え、モンテカルロ計算によって整合的な同時分布と特定シナリオの発生確率を導く。そこで生まれる結論は、相互依存を加味して修正された確率という型をとる。独立を仮定した確率の単純な積と対比させることで、論文は「事象どうしが連鎖するがゆえに、最悪のシナリオは見かけより起きやすい（あるいは起きにくい）」という、相互作用に根ざした言明にたどり着ける。

この手法が結論に持ち込む規定力の中心は境界設定にある。行列に組み込まれた事象しか相互作用の計算に入らず、列挙されなかった事象や、事象どうしの三体以上の高次の絡み合いは、整合化された分布の外にとどまる。時間観としては、未来を確率的に分岐する事象の束として捉え、その分岐が互いに条件づけ合うものと見る。これは単線の予測よりは豊かだが、未来を能動的な選択や設計の対象としてではなく、確率的に到来するものとして描く点で、なお決定論寄りの色を残す。

価値の所在という点で決定的なのは、初期確率とクロスインパクト行列の各要素がいずれも専門家の主観的判断だという事実である。行列のどの欄に強い促進関係を置くかが、修正後確率の符号と大きさを左右し、ひいては論文の結論を左右する。計算の精緻さは、入力された主観的判断に客観性の外観を与えてしまう危うさを伴う。したがってこの手法を用いた論文の頑健性は、行列要素への摂動に対する感度分析をどれだけ誠実に示すかにかかっており、それを欠けば結論は作成者の事前信念を数値で粉飾したものになりかねない。

## 発展課題

**課題A**: クロスインパクト行列の特定要素（例 `M[E1->E2]`）を 0.5〜5.0 で変化させ、E2 の修正後周辺確率がどう動くかを表にせよ。最も影響の大きい要素を特定し、専門家の合意形成で優先的に詰めるべき要素を論ぜよ。

**課題B**: 初期確率ベクトル `PRIOR` の各要素に推定誤差（±0.1 の摂動）を加えた複数バージョンで計算を回し、修正後確率とシナリオ確率のばらつきを集計せよ。初期確率の不確実性が結論をどれだけ揺らすかを評価せよ。